In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
pd.set_option("display.max_columns", None)

In [2]:
CSV_PATH = "../en.openfoodfacts.org.products 2.csv"

df = pd.read_csv(CSV_PATH, sep="\t", low_memory=False, on_bad_lines="skip")
df = df.copy()

# Helper: replace junk placeholder strings with NaN
INVALID = {"?", ".", ",", "n-a", "na", "none", "null", "0", "en:null", "en:none"}

def clean_series(s):
    s = s.str.strip()
    s = s.replace("", np.nan)
    s = s.where(~s.str.lower().isin(INVALID), np.nan)
    s = s.where(~s.str.fullmatch(r"[?,.\-/ ]+", na=False), np.nan)
    return s

_W = 68

def section_header(title):
    print(f"\n{'═' * _W}")
    print(f"  {title}")
    print(f"{'─' * _W}")

def merge_report(col_name, before, after):
    pct   = after / len(df) * 100
    added = after - before
    bar   = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"  {col_name:<22}  [{bar}]  {pct:5.1f}%   +{added:,}")


# Ozan: Bereinigung der Spalten brands, labels, countries, food_groups und main_category
section_header("Ozan  ·  brands · labels · countries · food_groups · main_category")

# Brands — fill missing brand names from the English and tag variants, then drop redundant columns
brands_before = df["brands"].notna().sum()
df["brands"] = df["brands"].fillna(df["brands_en"]).fillna(df["brands_tags"])
merge_report("brands", brands_before, df["brands"].notna().sum())
df = df.drop(columns=["brands_en", "brands_tags"])

# Labels — fill missing label info from the raw and tag variants, keep only the English column
labels_before = df["labels_en"].notna().sum()
df["labels_en"] = df["labels_en"].fillna(df["labels"]).fillna(df["labels_tags"])
merge_report("labels_en", labels_before, df["labels_en"].notna().sum())
df = df.drop(columns=["labels", "labels_tags"])

# Countries — fill missing country data from tag and raw variants, keep only the English column
countries_before = df["countries_en"].notna().sum()
df["countries_en"] = df["countries_en"].fillna(df["countries_tags"]).fillna(df["countries"])
merge_report("countries_en", countries_before, df["countries_en"].notna().sum())
df = df.drop(columns=["countries_tags", "countries"])

# Food Groups — fill missing food group values from tag and raw variants, keep only the English column
food_groups_before = df["food_groups_en"].notna().sum()
df["food_groups_en"] = df["food_groups_en"].fillna(df["food_groups_tags"]).fillna(df["food_groups"])
merge_report("food_groups_en", food_groups_before, df["food_groups_en"].notna().sum())
df = df.drop(columns=["food_groups_tags", "food_groups"])

# Main Category — fill missing main category from the raw variant, then drop the redundant column
main_category_before = df["main_category_en"].notna().sum()
df["main_category_en"] = df["main_category_en"].fillna(df["main_category"])
merge_report("main_category_en", main_category_before, df["main_category_en"].notna().sum())
df = df.drop(columns=["main_category"])


# Meme: Bereinigung der Spalten packaging, additives und ingredients
section_header("Meme  ·  packaging · additives · ingredients · manufacturing")

# Packaging — fill missing packaging info from raw, tag, and free-text variants; prefer structured English values
packaging_before = df["packaging_en"].notna().sum()
df["packaging_en"] = (
    df["packaging_en"]
    .fillna(df["packaging"])
    .fillna(df["packaging_tags"])
    .fillna(df["packaging_text"])
)
merge_report("packaging_en", packaging_before, df["packaging_en"].notna().sum())
df = df.drop(columns=["packaging", "packaging_tags", "packaging_text"])

# Additives — fill missing additive tags from the English name column, drop the now-redundant columns
additives_before = df["additives_tags"].notna().sum()
df["additives_tags"] = df["additives_tags"].fillna(df["additives_en"])
merge_report("additives_tags", additives_before, df["additives_tags"].notna().sum())
df = df.drop(columns=["additives_en", "additives"])

# Ingredients — fill missing ingredient tags from the free-text ingredient list, then drop the text column
ingredients_before = df["ingredients_tags"].notna().sum()
df["ingredients_tags"] = df["ingredients_tags"].fillna(df["ingredients_text"])
merge_report("ingredients_tags", ingredients_before, df["ingredients_tags"].notna().sum())
df = df.drop(columns=["ingredients_text"])

# Manufacturing Places — fill missing tag data from the raw text column, then drop the redundant column
manufacturing_before = df["manufacturing_places_tags"].notna().sum()
df["manufacturing_places_tags"] = df["manufacturing_places_tags"].fillna(df["manufacturing_places"])
merge_report("manufacturing_places_tags", manufacturing_before, df["manufacturing_places_tags"].notna().sum())
df = df.drop(columns=["manufacturing_places"])


# Bogomil: Bereinigung der Spalten states, manufacturing_places und categories
section_header("Bogomil  ·  states · categories")

# States — the English column is the most complete; drop the raw and tag duplicates and rename for clarity
df = df.drop(columns=["states", "states_tags"])
df = df.rename(columns={"states_en": "states"})

# drop allergens_en due to it having only 1 entries that is also invalid
df = df.drop(columns=["allergens_en"])

# drop emb_codes_tags
df = df.drop(columns=["emb_codes_tags"])

# Categories — clean junk values first, then merge into one column
# junk-cleaning is applied to all three variants before merging to avoid carrying over placeholder values
for c in ["categories", "categories_tags", "categories_en"]:
    df[c] = clean_series(df[c].astype(str).where(df[c].notna(), np.nan))

# Prefer the English variant, fall back to structured tags, then to the raw string
categories_before = df["categories_en"].notna().sum()
df["categories_final"] = (
    df["categories_en"]
    .fillna(df["categories_tags"])
    .fillna(df["categories"])
)
merge_report("categories_final", categories_before, df["categories_final"].notna().sum())
df = df.drop(columns=["categories", "categories_tags", "categories_en"])


# Deniz: Bereinigung der Spalten product_name, origins, cities, traces und quantity/serving
section_header("Deniz  ·  product_name · origins · cities · traces · quantity/serving")

# Product Name — fill from abbreviated name then generic name; both are dropped afterwards
product_name_before = df["product_name"].notna().sum()
df["product_name"] = df["product_name"].fillna(df["abbreviated_product_name"]).fillna(df["generic_name"])
merge_report("product_name", product_name_before, df["product_name"].notna().sum())
df = df.drop(columns=["abbreviated_product_name", "generic_name"])

# Origins — junk-clean all three variants first (many rows have '?', '-', 'none'),
# then prefer the English column, fall back to tags, then raw string
for c in ["origins", "origins_tags", "origins_en"]:
    df[c] = clean_series(df[c].astype(str).where(df[c].notna(), np.nan))

origins_before = df["origins_en"].notna().sum()
df["origins_en"] = df["origins_en"].fillna(df["origins_tags"]).fillna(df["origins"])
merge_report("origins_en", origins_before, df["origins_en"].notna().sum())
df = df.drop(columns=["origins", "origins_tags"])

# Cities — the `cities` column has exactly 1 non-null entry containing state metadata, not city data;
# drop it and keep the structured tag column
df = df.drop(columns=["cities"])

# Traces — fill missing values from the tag column then from the English variant; drop both afterwards
traces_before = df["traces"].notna().sum()
df["traces"] = df["traces"].fillna(df["traces_tags"]).fillna(df["traces_en"])
merge_report("traces", traces_before, df["traces"].notna().sum())
df = df.drop(columns=["traces_tags", "traces_en"])

# Quantity / Serving — free-text strings too unstructured to merge reliably;
# product_quantity and serving_quantity already hold parsed numeric values
df = df.drop(columns=["quantity", "serving_size"])


# Spalten umbenennen — standardise column names to short, consistent English labels
df = df.rename(columns={
    "labels_en":                 "labels",               # English label text
    "countries_en":              "countries",            # English country names
    "food_groups_en":            "food_groups",          # English food group names
    "main_category_en":          "main_category",        # English main category
    "packaging_en":              "packaging",            # English packaging description
    "additives_tags":            "additives",            # structured additive tags (e.g. en:e330)
    "ingredients_tags":          "ingredients",          # structured ingredient tags
    "manufacturing_places_tags": "manufacturing_places", # structured manufacturing location tags
    "origins_en":                "origins",              # English origin names
    "cities_tags":               "cities",               # structured city/location tags
})

print(f"\n{'─' * _W}")
print(f"  Zwischenstand: {df.shape[1]} Spalten gesamt  |  {len(df):,} Zeilen")
print(f"{'─' * _W}")




════════════════════════════════════════════════════════════════════
  Ozan  ·  brands · labels · countries · food_groups · main_category
────────────────────────────────────────────────────────────────────
  brands                  [████████████░░░░░░░░]   62.7%   +839
  labels_en               [█████░░░░░░░░░░░░░░░]   27.2%   +38
  countries_en            [███████████████████░]   99.5%   +5
  food_groups_en          [██████░░░░░░░░░░░░░░]   32.2%   +1
  main_category_en        [████████░░░░░░░░░░░░]   41.1%   +2

════════════════════════════════════════════════════════════════════
  Meme  ·  packaging · additives · ingredients · manufacturing
────────────────────────────────────────────────────────────────────
  packaging_en            [█░░░░░░░░░░░░░░░░░░░]    8.7%   +12,892
  additives_tags          [███░░░░░░░░░░░░░░░░░]   15.5%   +0
  ingredients_tags        [█████░░░░░░░░░░░░░░░]   28.5%   +3,299
  manufacturing_places_tags  [░░░░░░░░░░░░░░░░░░░░]    4.9%   +64

═══════════════